# Entrenamiento YOLOv8n-cls — Aguacatia
**Versión: v1.9 (2026-06-06)**

**Dataset:** Hass Avocado Ripening Photographic Dataset (Mendeley, CC BY 4.0)  
**Modelo:** YOLOv8n-cls (nano, clasificación de imágenes)  
**Clases:** 5 etapas de maduración  

## Instrucciones
1. **Activar GPU PRIMERO**: `Runtime → Disconnect and delete runtime` → `Change runtime type → T4 GPU` → Save → Reconectar
2. Ejecutar todo: `Ctrl + F9`
3. Si no hay GPU, el paso 6 detiene todo con instrucciones
4. Al finalizar (~45 min): `best.pt` queda en `Mi unidad/aguacatia/`

In [ ]:
_OK = {}  # Tracker global de pasos

def paso(n, titulo):
    print()
    print('━'*55)
    print(f'  PASO {n}/11 — {titulo}')
    print('━'*55)

def ok(n):
    _OK[n] = '✓'
    print(f'  ✓ PASO {n} COMPLETADO')
    print(f'  Estado: {_OK}')

def estado():
    print()
    print('━'*55)
    print('  ESTADO DEL NOTEBOOK')
    for i in range(1, 12):
        marca = _OK.get(i, '✗ PENDIENTE/FALLÓ')
        print(f'    PASO {i:2d}: {marca}')
    print('━'*55)

# ── PASO 1: Instalar dependencias ─────────────────────────
paso(1, 'Instalar dependencias')
try:
    !pip install -q ultralytics openpyxl scikit-learn matplotlib seaborn
    import ultralytics, numpy as np, torch, sklearn, PIL
    print(f'  ultralytics : {ultralytics.__version__}')
    print(f'  numpy       : {np.__version__}')
    print(f'  torch       : {torch.__version__}')
    ok(1)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(2, 'Montar Google Drive')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ok(2)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(3, 'Configuración')
try:
    import os
    from datetime import datetime

    ZIP_PATH    = '/content/drive/MyDrive/aguacatia/Hass Avocado Ripening Photographic Dataset.zip'
    DATASET_DIR = '/content/aguacatia_dataset'
    RAW_DIR     = f'{DATASET_DIR}/raw'
    PROC_DIR    = f'{DATASET_DIR}/processed'
    CLASSES     = ['unripe', 'breaking', 'ripe_first', 'ripe_second', 'overripe']
    EPOCHS      = 100
    IMG_SIZE    = 640
    BATCH       = 32
    PATIENCE    = 15

    from pathlib import Path
    zip_existe = Path(ZIP_PATH).exists()
    print(f'  ZIP_PATH  : {ZIP_PATH}')
    print(f'  ZIP existe: {"SI" if zip_existe else "NO — REVISAR RUTA"}')
    print(f'  Clases    : {CLASSES}')
    _START = datetime.now()
    ok(3)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(4, 'Extraer ZIP y organizar imágenes por clase')
try:
    import zipfile, shutil, openpyxl
    from pathlib import Path

    # Mapeo flexible — cubre números, texto en inglés y variantes comunes
    LABEL_MAP = {
        'unripe': 'unripe',         'breaking': 'breaking',
        'ripe': 'ripe_first',       'ripe (2)': 'ripe_second',
        'ripe_1': 'ripe_first',     'ripe_2': 'ripe_second',
        'ripe first': 'ripe_first', 'ripe second': 'ripe_second',
        'ripe 1': 'ripe_first',     'ripe 2': 'ripe_second',
        'ripe1': 'ripe_first',      'ripe2': 'ripe_second',
        'overripe': 'overripe',     'over ripe': 'overripe',
        '1': 'unripe', '2': 'breaking', '3': 'ripe_first',
        '4': 'ripe_second', '5': 'overripe',
        'stage 1': 'unripe', 'stage 2': 'breaking',
        'stage 3': 'ripe_first', 'stage 4': 'ripe_second', 'stage 5': 'overripe',
    }
    # Mapeo de nombres de carpetas a clases
    FOLDER_MAP = {
        'unripe': 'unripe', 'breaking': 'breaking',
        'ripe_first': 'ripe_first', 'ripe_second': 'ripe_second', 'overripe': 'overripe',
        'ripe first': 'ripe_first', 'ripe second': 'ripe_second',
        'ripe 1': 'ripe_first', 'ripe 2': 'ripe_second',
        'ripe1': 'ripe_first', 'ripe2': 'ripe_second',
        'stage1': 'unripe', 'stage2': 'breaking', 'stage3': 'ripe_first',
        'stage4': 'ripe_second', 'stage5': 'overripe',
        '1': 'unripe', '2': 'breaking', '3': 'ripe_first', '4': 'ripe_second', '5': 'overripe',
    }

    for cls in CLASSES:
        os.makedirs(f'{RAW_DIR}/{cls}', exist_ok=True)

    # Auto-detectar ZIP
    zip_path = Path(ZIP_PATH)
    if not zip_path.exists():
        matches = [z for z in Path('/content/drive/MyDrive').rglob('*.zip')
                   if any(k in z.name.lower() for k in ['avocado','hass','ripening','aguacatia'])]
        if not matches:
            raise FileNotFoundError(f'ZIP no encontrado en: {ZIP_PATH}')
        zip_path = matches[0]

    print(f'  Extrayendo {zip_path.name} ...')
    tmp = '/content/_tmp'
    if Path(tmp).exists(): shutil.rmtree(tmp)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(tmp)

    tmp_path = Path(tmp)

    # ── Estrategia 1: imágenes ya en subcarpetas por clase ─────────────────
    found = 0
    print('\n  Buscando estructura de carpetas...')
    for folder in tmp_path.rglob('*'):
        if not folder.is_dir(): continue
        folder_key = folder.name.strip().lower()
        cls = FOLDER_MAP.get(folder_key)
        if cls:
            imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.JPG'))
            for img in imgs:
                shutil.copy2(img, f'{RAW_DIR}/{cls}/{img.name}')
                found += 1
            if imgs:
                print(f'    Carpeta "{folder.name}" → {cls}: {len(imgs)} imgs')

    if found > 0:
        print(f'\n  Método: carpetas — {found} imágenes organizadas')
    else:
        # ── Estrategia 2: usar Excel ────────────────────────────────────────
        print('  Sin carpetas por clase — usando Excel...')
        xlsx = next(tmp_path.rglob('*.xlsx'), None)
        if not xlsx:
            raise FileNotFoundError('No se encontró Excel en el ZIP')

        images_dir = next((d for d in tmp_path.rglob('*')
                           if d.is_dir() and len(list(d.glob('*.jpg'))) > 100), None)
        if not images_dir:
            raise RuntimeError('No se encontró carpeta con JPGs en el ZIP')

        print(f'  Excel: {xlsx.name} | Imágenes: {len(list(images_dir.glob("*.jpg")))} JPGs')

        wb = openpyxl.load_workbook(xlsx, read_only=True, data_only=True)
        ws = wb.active
        all_rows = list(ws.iter_rows(values_only=True))
        headers = [str(c).strip().lower() if c else '' for c in all_rows[0]]

        print(f'\n  Columnas Excel: {headers}')
        print(f'  Primeras 3 filas: {all_rows[1:4]}')

        img_col = next((i for i, h in enumerate(headers)
                        if any(k in h for k in ['file','image','name','photo','img'])), 0)
        lbl_col = next((i for i, h in enumerate(headers)
                        if any(k in h for k in ['ripe','stage','label','class','matur'])),
                       len(headers)-1)

        print(f'  Col imagen=[{img_col}]"{headers[img_col]}" | Col etiqueta=[{lbl_col}]"{headers[lbl_col]}"')

        labels = {}
        unique_labels = set()
        for row in all_rows[1:]:
            if not row[img_col]: continue
            fn = str(row[img_col]).strip()
            if not fn.endswith('.jpg'): fn += '.jpg'
            raw = str(row[lbl_col]).strip().lower() if row[lbl_col] else ''
            unique_labels.add(raw)
            mapped = LABEL_MAP.get(raw)
            if mapped: labels[fn] = mapped
        wb.close()

        print(f'\n  Etiquetas únicas: {sorted(unique_labels)}')
        unmapped = sorted(unique_labels - set(LABEL_MAP.keys()))
        if unmapped:
            print(f'  SIN MAPEAR: {unmapped}')

        for fn, cls in labels.items():
            src = images_dir / fn
            if src.exists():
                shutil.copy2(src, f'{RAW_DIR}/{cls}/{fn}')
                found += 1

        if found == 0:
            raise RuntimeError(
                f'0 imágenes copiadas.\n'
                f'  Etiquetas en Excel : {sorted(unique_labels)}\n'
                f'  Pegar este output para ajustar el mapeo.'
            )

    shutil.rmtree(tmp)
    print('\n  Por clase:')
    total = 0
    for cls in CLASSES:
        n = len(list(Path(f'{RAW_DIR}/{cls}').glob('*.jpg')))
        total += n
        print(f'    {cls:15s}: {n}')
    print(f'    {"TOTAL":15s}: {total}')

    if total == 0:
        raise RuntimeError('Total 0 imágenes — ver diagnóstico arriba')

    ok(4)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(5, 'Split train / val / test 70/15/15')
try:
    import random, shutil
    from pathlib import Path

    random.seed(42)
    totals = {'train': 0, 'val': 0, 'test': 0}

    print(f'  {"Clase":15s} {"Train":>6} {"Val":>6} {"Test":>6} {"Total":>6}')
    print('  ' + '-'*43)
    for cls in CLASSES:
        imgs = sorted(Path(f'{RAW_DIR}/{cls}').glob('*.jpg'))
        random.shuffle(imgs)
        n = len(imgs)
        n_train = int(n * 0.70)
        n_val   = int(n * 0.15)
        splits  = {'train': imgs[:n_train],
                   'val':   imgs[n_train:n_train+n_val],
                   'test':  imgs[n_train+n_val:]}
        counts = {}
        for split, split_imgs in splits.items():
            dest = Path(f'{PROC_DIR}/{split}/{cls}')
            dest.mkdir(parents=True, exist_ok=True)
            for img in split_imgs:
                shutil.copy2(img, dest / img.name)
            counts[split] = len(split_imgs)
            totals[split] += len(split_imgs)
        print(f'  {cls:15s} {counts["train"]:>6} {counts["val"]:>6} {counts["test"]:>6} {n:>6}')

    grand = sum(totals.values())
    print('  ' + '-'*43)
    print(f'  {"TOTAL":15s} {totals["train"]:>6} {totals["val"]:>6} {totals["test"]:>6} {grand:>6}')
    ok(5)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(6, 'Verificar GPU')
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f'  CUDA disponible : {cuda_ok}')
    if cuda_ok:
        print(f'  GPU             : {torch.cuda.get_device_name(0)}')
        print(f'  VRAM            : {round(torch.cuda.get_device_properties(0).total_memory/1e9,1)} GB')
        _DEVICE = 0
        ok(6)
    else:
        print()
        print('  ✗ NO HAY GPU — DETENIÉNDO EJECUCIÓN')
        print()
        print('  SOLUCIÓN:')
        print('  1. Runtime → Disconnect and delete runtime')
        print('  2. Runtime → Change runtime type → T4 GPU o H100 → Save')
        print('  3. Reconectar → Ctrl+F9')
        print()
        raise RuntimeError('Sin GPU. Seguir en CPU tomaría +100 horas. Cambiar runtime primero.')
except RuntimeError:
    raise
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(7, 'Entrenar YOLOv8n-cls (~30-60 min)')
try:
    import os
    from ultralytics import YOLO
    from pathlib import Path
    from datetime import datetime

    # Desactivar wandb — debe estar antes de cualquier llamada a YOLO
    os.environ['WANDB_DISABLED'] = 'true'
    os.environ['WANDB_MODE']     = 'disabled'
    os.environ['WANDB_SILENT']   = 'true'
    try:
        import wandb
        wandb.init(mode='disabled')
    except Exception:
        pass

    # Cambiar a /content para que 'project' sea un nombre simple sin barras
    os.chdir('/content')

    train_imgs = list(Path(PROC_DIR).glob('train/**/*.jpg'))
    if not train_imgs:
        raise RuntimeError(f'0 imágenes en {PROC_DIR}/train/ — ejecutar pasos 4 y 5 primero')
    print(f'  Imágenes train : {len(train_imgs)}')
    print(f'  Inicio         : {datetime.now().strftime("%H:%M:%S")}')

    model   = YOLO('yolov8n-cls.pt')
    results = model.train(
        data=PROC_DIR, epochs=EPOCHS, imgsz=IMG_SIZE,
        batch=BATCH, patience=PATIENCE, device=_DEVICE,
        project='aguacatia_runs',          # nombre simple, sin barras
        name='yolov8n_cls_aguacatia', exist_ok=True,
        augment=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        flipud=0.1, fliplr=0.5,
    )
    BEST_PT = str(Path(results.save_dir) / 'weights' / 'best.pt')
    print(f'  Modelo guardado : {BEST_PT}')
    ok(7)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    estado()
    raise

In [ ]:
paso(8, 'Evaluar en test set')
try:
    import numpy as np
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
    from pathlib import Path

    best_model = YOLO(BEST_PT)
    y_true, y_pred, y_proba = [], [], []

    for cls_idx, cls in enumerate(CLASSES):
        imgs = list(Path(f'{PROC_DIR}/test/{cls}').glob('*.jpg'))
        print(f'  {cls:15s}: {len(imgs)} imágenes')
        for img_path in imgs:
            result = best_model.predict(str(img_path), verbose=False)[0]
            probs = result.probs.data.cpu().numpy()
            y_true.append(cls_idx)
            y_pred.append(int(np.argmax(probs)))
            y_proba.append(probs)

    y_true  = np.array(y_true)
    y_pred  = np.array(y_pred)
    y_proba = np.array(y_proba)
    accuracy = (y_true == y_pred).mean()

    print(f'\n  Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    ok(8)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(9, 'Matriz de confusión')
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm,      annot=True, fmt='d',   cmap='YlOrRd', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
    axes[0].set_title('Absolutos');    axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicción')
    axes[1].set_title('Normalizada'); axes[1].set_ylabel('Real'); axes[1].set_xlabel('Predicción')
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    ok(9)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(10, 'Curvas ROC y AUC')
try:
    from sklearn.preprocessing import label_binarize

    y_true_bin = label_binarize(y_true, classes=list(range(len(CLASSES))))
    fig, ax    = plt.subplots(figsize=(9, 7))
    colors     = ['#22c55e','#eab308','#f97316','#a855f7','#ef4444']
    auc_scores = []

    for i, (cls, color) in enumerate(zip(CLASSES, colors)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        auc = roc_auc_score(y_true_bin[:, i], y_proba[:, i])
        auc_scores.append(auc)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC={auc:.3f})')
        print(f'  {cls:15s}: AUC={auc:.4f}')

    print(f'  AUC macro      : {np.mean(auc_scores):.4f}')
    ax.plot([0,1],[0,1],'k--',lw=1); ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('Falsos Positivos'); ax.set_ylabel('Verdaderos Positivos')
    ax.set_title(f'Curvas ROC — AUC macro={np.mean(auc_scores):.3f}')
    ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    ok(10)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

In [ ]:
paso(11, 'Guardar en Google Drive')
try:
    import shutil, os

    drive_dest = '/content/drive/MyDrive/aguacatia/'
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(BEST_PT,                         f'{drive_dest}/best.pt')
    shutil.copy('/content/confusion_matrix.png', f'{drive_dest}/confusion_matrix.png')
    shutil.copy('/content/roc_curves.png',       f'{drive_dest}/roc_curves.png')
    ok(11)
except Exception as e:
    print(f'  ✗ FALLÓ: {type(e).__name__}: {e}')
    raise

estado()

print()
print('━'*55)
print('  RESUMEN FINAL')
print(f'  Accuracy : {accuracy*100:.1f}%')
print(f'  AUC macro: {np.mean(auc_scores):.3f}')
elapsed = datetime.now() - _START
print(f'  Tiempo   : {str(elapsed).split(".")[0]}')
print('━'*55)
print()
print('  SIGUIENTE PASO:')
print('  1. Descargar best.pt de Google Drive (Mi unidad/aguacatia/)')
print('  2. Copiarlo como api/model/best.pt en el repo')
print('  3. git add api/model/best.pt && git push')
print('  → Coolify reconstruye la API automáticamente')